In [1]:
from easydict import EasyDict as edict
#from train import train_model
#from test import test_model
from monai.utils import set_determinism
import torch
from monai.transforms import (
    EnsureChannelFirstd,
    Compose,
    LoadImaged,
    ScaleIntensityD,
    EnsureTypeD,
    Resized,
    RandSpatialCropd,
    SpatialPadd,
    Orientationd,
    ScaleIntensityRanged,
    Spacingd,
    DeleteItemsd
)
from monai.data import PersistentDataset, DataLoader
from glob import glob
import random
import pandas as pd
from monai.data import ITKReader
import json 
with open("pipeline_test.json", "r") as f:
    cfg = edict(json.load(f))


In [2]:
df = pd.read_csv(cfg.train_csv)
filepaths = [{"im": fname} for fname in df['Path'][:50]]
filepaths = [{"im": p['im'].replace('D:/Backup/Mestrado/dados/', '/home/jovyan/imgs/').replace('\\', '/')} for p in filepaths]
#filepaths = [{"im": p['im'].replace('D:/Backup/Mestrado/dados/', '/home/jovyan/imgs/')} for p in filepaths]
filepaths = [{"im": p['im'].replace('D:/Mestrado/dados/', '/home/jovyan/imgs/').replace('\\', '/')} for p in filepaths]
#filepaths = [{"im": p['im'].replace('D:/Mestrado/dados/', '/home/jovyan/imgs/')} for p in filepaths]

In [2]:
cache_dir = 'D:/Backup/Mestrado/proadi.neuro.monai.bundle.anomalia/cache_dir'

In [6]:
from dataset import get_dataloader
from tqdm import tqdm

train_dl, val_dl = get_dataloader(cfg, mode = 'train')
for data in tqdm(train_dl):
        img = data['im'].to(cfg.device)
#         # ==========forward=========
#         if img.size() != torch.Size([1, 1, 96, 96, 96]):
#             print(img.size())
#             break

total number of images: 50
number of images for training: 40
number of images for validation: 40


  0%|                                                    | 0/40 [00:00<?, ?it/s]You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
ImageSeriesReader (0x60e9b52166e0): Non uniform sampling or miss

In [5]:
# Já é o SPR sem arquivos com '_label'
df = pd.read_csv('data/quali_dataset_sample_back.csv')

df_train = df[df['Sample'] == 'train'][:50]
# filepaths = [{"im": fname} for fname in df['Path']]
# random.shuffle(filepaths)


df_train.head()

,patient_id,study_id,dataset_id,patient_sex,age,machine,number_of_slices,slice_thickness,pixel_spacing_xy,pixel_spacing_z,rescale_slope,rescale_intercept,Normal,Path,png_Path,Sample
0,2634,"1,28E+73",jpr_dados,F,77.0,GE MEDICAL SYSTEMS,139,2.50,0.406250,1.250,1.0,-1024.0,True,D:/Backup/Mestrado/dados/JPR/dataset_jpr_train...,E:/unsupervised/unsupervised2/Unsupervised-Les...,train
1,2635,"1,28E+73",jpr_dados,F,68.0,GE MEDICAL SYSTEMS,139,2.50,0.406250,1.250,1.0,-1024.0,True,D:/Backup/Mestrado/dados/JPR/dataset_jpr_train...,E:/unsupervised/unsupervised2/Unsupervised-Les...,train
2,2453,"1,28E+73",jpr_dados,F,60.0,GE MEDICAL SYSTEMS,145,2.50,0.488281,1.250,1.0,-1024.0,True,D:/Backup/Mestrado/dados/JPR/dataset_jpr_train...,E:/unsupervised/unsupervised2/Unsupervised-Les...,train
3,2522,"1,28E+73",jpr_dados,F,36.0,GE MEDICAL SYSTEMS,265,1.25,0.562500,0.625,1.0,-1024.0,True,D:/Backup/Mestrado/dados/JPR/dataset_jpr_train...,E:/unsupervised/unsupervised2/Unsupervised-Les...,train
4,2432,"1,28E+72",jpr_dados,M,56.0,TOSHIBA,226,1.00,0.430000,0.800,1.0,0.0,True,D:/Backup/Mestrado/dados/JPR/dataset_jpr_train...,E:/unsupervised/unsupervised2/Unsupervised-Les...,train


In [23]:
# Essas transformações foram feitas em um treinamento antigo
# transforms = Compose(
#     [
#         LoadImageD(keys=["im"], image_only=False),
#         EnsureChannelFirstd(keys=["im"]),
#         ScaleIntensityD(keys=["im"]),
#         EnsureTypeD(keys=["im"]),
#     ]
# )

# novas transformações usadas também nas fatias das imagens para a rede VAE
transforms = Compose([
    LoadImaged(keys=["image"], image_only=False, reader=ITKReader(series_name="")),
    #LoadImaged(keys=["image"]),
    EnsureChannelFirstd(keys=["image"]),
    Orientationd(keys=["image"], axcodes="RAS"),
    ScaleIntensityRanged(
    keys=["image"],
    a_min=-15,
    a_max=85,
    b_min=0.0,
    b_max=1.0,
    clip=True
),
    Spacingd(
        keys=["image"],
        pixdim=(1.0, 1.0, 1.0),
        mode="bilinear"
    ),
    Resized(keys = ['image'], spatial_size = (160,192,96)),
    DeleteItemsd(keys=["image_meta_dict"]),  # drop all metadata entirely
    
    
    
])




#IXIT2_filenames = glob(cfg.train_csv + "/*.nii.gz")

#IXIT2_filenames = glob(cfg.valid_csv + "/*.nii.gz")
#random.shuffle(df['Path'])


monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.


In [32]:
# Já é o SPR sem arquivos com '_label'
df = pd.read_csv('data/quali_dataset_sample_back.csv')

df_train = df[(df['Sample'] == 'train')][:50]
train_datadict = [{"image": fname} for fname in df_train['Path']]
# random.shuffle(filepaths)


# Split into training and testing
# test_frac = 0.2
# num_ims = len(filepaths)
# num_test = int(num_ims * test_frac)
# num_train = num_ims - num_test

# num_train = 500

# train_datadict = filepaths[:num_train]
# val_datadict = filepaths[-num_test:]
# print(f"total number of images: {num_ims}")
# print(f"number of images for training: {len(train_datadict)}")
# print(f"number of images for testing: {len(val_datadict)}")


In [33]:

### List images without _label in the spr directory
# path = "/home/validacao-retrospectiva-neuro-new/spr-head-ct-age-prediction-challenge/dataset_jpr_train/dataset_jpr_train"

# files = []
# for item in os.listdir(path):
#     numbered_folders = os.path.join(path, item)
#     print(numbered_folders)
#     for image in os.listdir(numbered_folders):
#         if "_label" in image:
#             pass
#         else:
#             files.append(os.path.join(numbered_folders,image))
# import pandas as pd


# df = pd.DataFrame(files)
# df['Source'] = 'SPR'
# df.rename(columns = {0: 'Path'}, inplace = True)
# df.to_csv( 'spr_path.csv', index_label = False)

In [34]:

from monai.data import pad_list_data_collate

train_ds = PersistentDataset(train_datadict, transforms, cache_dir = cache_dir)
train_loader = DataLoader(train_ds, batch_size=10, num_workers=0, shuffle=True)


In [35]:
from tqdm import tqdm


In [36]:
for data in tqdm(train_loader):
        #print(data['image'])
        img = data['image']

100%|██████████| 5/5 [06:24<00:00, 76.98s/it]


In [10]:
import matplotlib.pyplot as plt
import torch

def visualize_batch(data, num_images=2, num_slices=5):
    """
    Visualize axial slices from a MONAI batch.
    data['image']: (B, C, H, W, D)
    """
    images = data['image']  # (B, C, H, W, D)
    
    if isinstance(images, torch.Tensor):
        images = images.numpy()
    
    B = min(num_images, images.shape[0])
    
    for b in range(B):
        img = images[b, 0]  # (H, W, D) — take first channel
        D = img.shape[2]
        
        # Pick evenly spaced slices
        slice_indices = [int(D * i / (num_slices + 1)) for i in range(1, num_slices + 1)]
        
        fig, axes = plt.subplots(1, num_slices, figsize=(4 * num_slices, 4))
        fig.suptitle(f"Sample {b+1} | Shape: {img.shape}", fontsize=14)
        
        for i, sl in enumerate(slice_indices):
            axes[i].imshow(img[:, :, sl], cmap="gray", origin="lower")
            axes[i].set_title(f"Slice {sl}")
            axes[i].axis("off")
        
        plt.tight_layout()
        plt.show()

# Usage inside your loop
for data in tqdm(train_loader):
    visualize_batch(data, num_images=2, num_slices=5)
    break

  0%|          | 0/32 [01:32<?, ?it/s]


AttributeError: 'Tensor' object has no attribute 'meta'